# 🚀 Visualisierung der Ergebnisse der Einstellungserkennung

Dieses Jupyter Notebook erstellt aus den Ergebnissen der <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html" class="external-link" target="_blank">automatisierten Einstellungserkennung</a> eine **interaktive Visualisierung zum Vergleich mehrerer Filme**.

Aus den Metadaten der Einstellungserkennung entstehen sechs interaktive Balkendiagramme. Sie zeigen:

- die **durchschnittliche und mediane Einstellungslänge**,
- die **längste und kürzeste erkannte Einstellung** sowie
- die **Filmlaufzeit und die Anzahl der erkannten Einstellungen**.

Für jedes Diagramm können die Filme unabhängig voneinander nach den Werten, alphabetisch oder chronologisch sortiert werden. So lassen sich unterschiedliche Aspekte der Einstellungserkennung miteinander vergleichen. Als Ergebnis entstehen eine Darstellung direkt im Notebook und eine `.html`-Datei, die sich im Browser öffnen und in Websites einbetten lässt.


Der Arbeitsablauf gliedert sich in **Festlegen der Parameter**, **technische Vorbereitung**, **Einlesen der Metadaten und Erstellung der Visualisierung** sowie **Export der Ergebnisse**. Die Texte erläutern vor allem die Funktion der Arbeitsschritte, ergänzende Kommentare im Python-Code beschreiben ihre technische Umsetzung.

Weitere Informationen zum Erkennungsverfahren und den erzeugten Metadaten finden Sie im <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html" class="external-link" target="_blank">Einführungskapitel zur Einstellungserkennung</a>.

## Ausführung des Notebooks

In der Jupyter-Book-Version der OER sind die Ergebnisse der Ausführung bereits gespeichert. Sie können daher betrachtet werden, ohne den Code selbst auszuführen. ***Für ein tieferes Verständnis der Lerninhalte und der Funktionen des Codes und für die Bearbeitung der späteren Übungen empfehlen wir jedoch, das Notebook in einer Cloud-Umgebung oder lokal auszuführen***.

**Cloud-Umgebung:** Klicken Sie oben rechts in der Menüleiste auf das Raketen-Symbol 🚀 und wählen Sie „Colab“, um das Notebook auszuführen. Eine GPU ist für dieses Notebook nicht erforderlich.

**Lokale Umgebung:** Klicken Sie oben rechts in der Menüleiste auf das Download-Symbol ↓ und wählen Sie „.ipynb“, um das Notebook herunterzuladen und lokal auszuführen. Abhängig von Browser und Betriebssystem müssen Sie die Datei gegebenenfalls per Rechtsklick herunterladen.

***Wir empfehlen außerdem, die Codeblöcke einzeln und nacheinander auszuführen, statt Funktionen wie „Alle ausführen“ in Colab oder „Run All Cells“ in anderen Anwendungen zu verwenden***. Die separate Ausführung der Codeblöcke erleichtert das Verständnis der damit verbundenen Funktionen und stellt sicher, dass alle vorbereitenden Schritte vor den nachfolgenden Codeblöcken ausgeführt wurden.

## Festlegen der Parameter

Zunächst werden die zentralen Einstellungen für den Import und die Darstellung der Daten festgelegt. Diese Parameter sind im Code großgeschrieben und können angepasst werden, ohne den späteren Code verändern zu müssen. Für einen ersten Probelauf empfiehlt es sich, die voreingestellten Werte beizubehalten.

`CSV_PATH` legt den Ordner fest, in dem sich die Metadaten-CSV-Dateien befinden. Diese entstehen beim Export des <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellungen_notebook.html" class="external-link" target="_blank">Notebooks zur Einstellungserkennung</a>. 

Mit `CSV_PATTERN` wird festgelegt, welche Dateinamen bei der Suche berücksichtigt werden. Das voreingestellte Muster `metadata` passt zu den exportierten Metadatendateien und muss normalerweise nicht geändert werden.

`EXPORT_PATH` bezeichnet den Ordner für die fertige HTML-Visualisierung. Falls die Eingabe- oder Exportordner noch nicht existieren, werden sie automatisch angelegt.

Für die **Darstellung der Balkendiagramme** können Sie zwei Werte anpassen:

- `CHART_WIDTH` bestimmt die Breite der Diagramme.
- `ROW_HEIGHT` legt fest, wie viel Platz jede Filmzeile einnimmt.

Mit `DOWNLOAD_TESTFILES` lässt sich der Download der bereitgestellten Testdaten aktivieren oder deaktivieren. Standardmäßig ist der Wert auf `True` gesetzt, sodass die Testdateien heruntergeladen werden. Für eigene Daten sollte der Wert auf `False` gesetzt werden. Über den Parameter `CUSTOM_ZIP_LINK` lässt sich alternativ ein eigenes Archiv über einen direkten Download-Link einbinden. Den Namen dieses Archivs legt `CUSTOM_ARCHIVE_NAME` fest.

In [1]:
# Creating the input and export directories.
import os

# Path to the directory containing one metadata CSV per film.
# Relative paths are resolved from the location where the notebook is executed.
CSV_PATH = "vis_files_shots"
# Match filenames containing "metadata", including files in subfolders.
CSV_PATTERN = "metadata"

# Path to the directory for the exported visualisation.
EXPORT_PATH = "export_vis_shots"

# Width of each plot in pixels, excluding the film labels on the left.
CHART_WIDTH = 500
# Height of each film row in pixels.
ROW_HEIGHT = 30

# If True, provided test metadata files are downloaded and extracted.
# Set this option to False when using your own files.
DOWNLOAD_TESTFILES = True

# Add your own download link inside quotes to load an additional ZIP archive.
CUSTOM_ZIP_LINK = "insert link here"
# An archive with the same name is reused instead of downloaded again.
CUSTOM_ARCHIVE_NAME = "custom_archive"

# Create both directories. exist_ok=True allows them to exist already.
os.makedirs(CSV_PATH, exist_ok=True)
os.makedirs(EXPORT_PATH, exist_ok=True)

---
## Technische Vorbereitung

Bevor die Visualisierung erstellt werden kann, werden die benötigten Programmbibliotheken installiert und importiert. Anschließend können Testdateien oder eigene Metadatendateien heruntergeladen werden.

### Installation der Programmbibliotheken

Der folgende Codeblock installiert die benötigten Python-Bibliotheken. Mit `pandas` lassen sich die Metadaten aus mehreren CSV-Dateien in einer gemeinsamen Tabelle zusammenführen und bearbeiten. `Altair` ist eine Python-Bibliothek zur Erstellung von Diagrammen aus tabellarischen Daten.

Anders als die Notebooks zur <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellungen_notebook.html" class="external-link" target="_blank">Einstellungserkennung</a> und <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/figuren_notebook.html" class="external-link" target="_blank">Figurenerkennung</a> benötigt dieses Notebook keine <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/technik.html#verwendete-hardware" class="external-link" target="_blank">GPU-Beschleunigung</a>. Da hier keine Videodateien, sondern lediglich CSV-Dateien mit Metadaten ausgewertet werden, ist durch eine GPU keine wahrnehmbare Beschleunigung zu erwarten.

In [9]:
print("Installing dependencies.")

# Install all third-party dependencies once in the active notebook kernel.
!pip install -q pandas 
!pip install -q altair==6.0.0

print("Dependencies installed.")

Installing dependencies.
Dependencies installed.


### Import der Programmbibliotheken

Durch den Import stehen die benötigten Bibliotheken in der aktuellen Notebook-Sitzung zur Verfügung. Sie werden für den Umgang mit Dateien, das Einlesen der Metadaten, die Erstellung der Diagramme und den späteren Export verwendet. Die einzelnen Aufgaben sind im Code kommentiert.

In [3]:
# Handling file paths when reading metadata and exporting the visualisation.
from pathlib import Path

# Downloading files from a URL and extracting ZIP archives.
from urllib.request import urlretrieve
import zipfile

# Generating unique HTML IDs so separate chart outputs do not interfere.
from uuid import uuid4
import sys

# Reading metadata tables and creating interactive charts.
import pandas as pd
import altair as alt

# Displaying the visualisation within the notebook.
from IPython.display import HTML, display

# Import the browser download function only within a Google Colab runtime.
colab = "google.colab" in sys.modules
if colab:
    from google.colab import files
else:
    print("Using non-Colab environment.")

print("Dependencies imported.")

Using non-Colab environment.
Dependencies imported.


### Download des Analysematerials

Für die Visualisierung werden **nur die Metadaten-CSV-Dateien** aus der Einstellungserkennung benötigt, nicht die ursprünglichen Videodateien oder die einzelnen Einstellungsannotationen. Das Notebook kann hierfür ein `.zip`-Archiv mit Testdateien herunterladen oder ein eigenes Archiv über die zuvor festgelegten Parameter laden. Die darin enthaltenen Dateien werden in den `CSV_PATH`-Ordner entpackt.

Bei den Testdateien handelt es sich um Einstellungsmetadaten zu allen Spielfilmen, bei denen <a href="https://www.filmportal.de/person/konrad-wolf_763849fa3ea84b50a6d3ef1da3fee547" class="external-link" target="_blank">Konrad Wolf</a> Regie geführt hat. Bei der Einstellungserkennung wurde ein <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellungen_notebook.html#festlegung-des-schwellenwerts" class="external-link" target="_blank">Schwellenwert</a> von `0.5` verwendet und die Verarbeitung auf einer <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/technik.html#verwendete-hardware" class="external-link" target="_blank">CUDA-fähigen GPU</a> durchgeführt. Ist `DOWNLOAD_TESTFILES = True` eingestellt, werden diese Daten automatisch heruntergeladen.

Wenn Sie eigene Analyseergebnisse verwenden möchten, können Sie den Testdatei-Download deaktivieren und die Metadaten-CSV-Dateien selbst im Eingabeordner oder einem seiner Unterordner ablegen.


In [ ]:
# Define a reusable function for downloading and extracting ZIP archives.
def download_zip(url: str, archive_name: str, target_directory: str):
    # Store the archive in the notebook's working directory.
    archive_path = Path(f"{archive_name}.zip")

    # Download only if an archive with the same name is not already present.
    if not archive_path.is_file():
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
    else:
        print(f"{archive_path} already downloaded.")

    # Extract the files into the selected metadata directory.
    with zipfile.ZipFile(archive_path) as zip_ref:
        zip_ref.extractall(target_directory)
    print(f"Extracted {archive_path} to {target_directory}.")


# Download and extract the test metadata if the setting is enabled.
if DOWNLOAD_TESTFILES:
    download_zip(
        "https://owncloud.gwdg.de/index.php/s/1ACZETeYPyIIy0w/download",
        "vis_files_shots", CSV_PATH,
    )
else:
    print("No testfile download requested.")

# Download and extract the custom archive if a link is provided.
if CUSTOM_ZIP_LINK != "insert link here":
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, CSV_PATH)

Extracted vis_files_shots.zip to vis_files_shots.


---
## Einlesen der Metadaten und Erstellung der Visualisierung

Im nächsten Schritt werden die Metadaten der einzelnen Filme zusammengeführt und für die Visualisierung vorbereitet. Anschließend erzeugt das Notebook aus dieser gemeinsamen Datengrundlage mehrere interaktive Balkendiagramme.

### Import und Transformation der Metadaten

Die Metadaten-CSV-Dateien speichern ihre Angaben als **Schlüssel-Wert-Paare**, also mit einem Metadatennamen und dem dazugehörigen Wert in jeder Zeile. Eine Datei kann beispielsweise so aussehen:

```csv
video_name,Goya_1971.mp4
title,Goya
release_year,1971
frame_count,192858
frame_height,470
frame_width,720
frame_rate,25.0
duration_hh_mm_ss,2:08:34
model,transnetv2
device,cuda
analysis_timestamp,2026-10-28T12:56:46+00:00
processing_time_hh_mm_ss,0:01:13
threshold,0.5
shot_count,1074
mean_duration_s,7.18
median_duration_s,3.76
max_duration_s,134.64
min_duration_s,0.32
min_prediction_score,0.0
max_prediction_score,1.0
```

Eine Erläuterung dieser einzelnen Datenpunkte finden Sie erneut im OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#filmname-shots-threshold-metadata-csv" class="external-link" target="_blank">zur Ausgabe der Einstellungserkennung.</a>

Für den Vergleich mehrerer Filme sucht das Notebook im Ordner `CSV_PATH` nach den durch `CSV_PATTERN` festgelegten Metadatendateien. Jede Datei enthält Angaben zu einem Film, beispielsweise den Filmtitel (`title`), das Erscheinungsjahr (`release_year`), die Anzahl erkannter Einstellungen (`shot_count`) und verschiedene Kennwerte zur Einstellungslänge. Die Ergebnisse dieser Suche werden in einer gemeinsamen Tabelle gespeichert.

Für die Visualisierung werden sechs Kennwerte ausgewählt: die durchschnittliche, mediane, längste und kürzeste Einstellungslänge sowie die Filmlaufzeit und die Anzahl der erkannten Einstellungen. Die Auswahl dieser Kennwerte und deren Benennung innerhalb der Visualisierung sind im Verzeichnis `metrics` festgelegt. Wenn Sie möchten, können Sie die Bezeichnungen und die Reihenfolge der Diagramme dort ändern.

Damit die Diagramme richtig sortiert und beschriftet werden, bereitet das Notebook die Werte noch auf. Beispielsweise wird die Filmlaufzeit aus Stunden, Minuten und Sekunden in **Minuten** umgerechnet. Außerdem werden die Filme alphabetisch und nach Erscheinungsjahr eingeordnet. Zuletzt werden für die Anzeige beim Bewegen des Mauszeigers über einen Balken die Zahlen durch passende Einheiten – `Sekunden`, `Minuten` oder `Einstellungen` – ergänzt. 

**Hinweis:** Die CSV-Dateien müssen die im Code verwendeten Metadatenfelder enthalten. Das Erscheinungsjahr (`release_year`) wird für die chronologische Sortierung benötigt und muss als Zahl vorliegen.


In [5]:
# Initialise the list of metadata files.
metadata_files = []
# Find metadata files in CSV_PATH matching CSV_PATTERN.
for csv_path in Path(CSV_PATH).rglob(f"*{CSV_PATTERN}*.csv"):
    # Check whether the path refers to a file.
    if csv_path.is_file():
        metadata_files.append(csv_path)

# Raise an error if no metadata files are found.         
if not metadata_files:
    raise FileNotFoundError(f"No files matching {CSV_PATTERN} in {CSV_PATH}.")

# Assign a German chart title to each metric. The order determines the chart order.
metrics = {
    "mean_duration_s": "Durchschnittliche Einstellungslänge in Sekunden",
    "median_duration_s": "Mediane Einstellungslänge in Sekunden",
    "max_duration_s": "Längste Einstellung in Sekunden",
    "min_duration_s": "Kürzeste Einstellung in Sekunden",
    "runtime_min": "Filmlaufzeit in Minuten",
    "shot_count": "Anzahl der Einstellungen",
}

# Read the metadata CSV files and store their values in a list.
# Initialise the list.
metadata_records = []
for csv_file in metadata_files:
    # Read each metadata CSV file as a Series.
    metadata = pd.read_csv(csv_file, header=None, index_col=0).iloc[:, 0]
    # Add the metadata Series to the list.
    metadata_records.append(metadata)
    print(f"Loaded metadata for {metadata['title']} from {csv_file.name}.")

# Build a DataFrame with one row per metadata file.
films = pd.DataFrame(metadata_records)

# Convert the release year to an integer.
films["release_year"] = films["release_year"].astype(int)

# Convert the metric values to decimal numbers.
for metric in metrics:
    # Runtime in minutes is calculated later, so skip it here.
    if metric == "runtime_min":
        continue
    films[metric] = films[metric].astype(float)

# Store the alphabetical sort order:
# Create lowercase titles for sorting without considering capitalisation.
films["sort_title"] = films["title"].str.casefold()
# Sort the films alphabetically.
films = films.sort_values("sort_title")
# Replace the previous row numbers with consecutive numbers.
films = films.reset_index(drop=True)
# Store the alphabetical position of each film.
films["alpha_rank"] = films.index

# Store the chronological position, starting with the oldest film.
# For identical years, keep the alphabetical order established above.
films["year_rank"] = films["release_year"].rank(method="first")

# Convert the runtime to minutes for visualisation.
films["runtime_min"] = pd.to_timedelta(films["duration_hh_mm_ss"]).dt.total_seconds() / 60

# Prepare formatted hover values for each metric.
for metric in metrics:
    # Initialise the list of hover values.
    hover_values = []

    for value in films[metric]:
        # Display shot counts without decimal places.
        if metric == "shot_count":
            number = f"{int(value):,}"
        else:
            number = f"{value:,.2f}"

        # Convert English number separators to German separators.
        number = number.translate(str.maketrans(",.", ".,"))
        
        # Add the appropriate unit.
        if metric == "runtime_min":
            number = number + " Minuten"
        elif metric == "shot_count":
            number = number + " Einstellungen"
        else:
            number = number + " Sekunden"

        hover_values.append(number)

    # Add the hover values to the film DataFrame.
    films[f"{metric}_hover"] = hover_values

Loaded metadata for Sonnensucher from Sonnensucher_1958_shots_0.5_metadata.csv.
Loaded metadata for Professor Mamlock from Professor Mamlock_1961_shots_0.5_metadata.csv.
Loaded metadata for Leute mit Flügeln from Leute mit Flügeln_1960_shots_0.5_metadata.csv.
Loaded metadata for Genesung from Genesung_1956_shots_0.5_metadata.csv.
Loaded metadata for Solo Sunny from Solo Sunny_1980_shots_0.5_metadata.csv.
Loaded metadata for Der nackte Mann auf dem Sportplatz from Der nackte Mann auf dem Sportplatz_1974_shots_0.5_metadata.csv.
Loaded metadata for Einmal ist keinmal from Einmal ist keinmal_1955_shots_0.5_metadata.csv.
Loaded metadata for Mama ich lebe from Mama ich lebe_1976_shots_0.5_metadata.csv.
Loaded metadata for Der geteilte Himmel from Der geteilte Himmel_1964_shots_0.5_metadata.csv.
Loaded metadata for Goya from Goya_1971_shots_0.5_metadata.csv.
Loaded metadata for Der kleine Prinz from Der kleine Prinz_1966_shots_0.5_metadata.csv.
Loaded metadata for Lissy from Lissy_1957_shots_

### Erstellung der interaktiven Visualisierung

Für die Erstellung der sechs Diagramme wird die Python-Bibliothek `Altair` verwendet. Jedes Diagramm stellt einen anderen Kennwert dar. Die Balken stehen für die jeweiligen Filme, ihre Länge zeigt den Wert des ausgewählten Merkmals. Unterschiedliche Farben erleichtern die Unterscheidung der sechs Diagramme.

`Altair` ermöglicht es, Diagramme aus tabellarischen Daten zu erstellen und mit interaktiven Funktionen auszustatten. Eine Einführung bietet die offizielle <a href="https://altair-viz.github.io/getting_started/starting.html" class="external-link" target="_blank">Altair-Dokumentation</a>.


#### Sortierung

Unter jedem Diagramm befindet sich eine eigene Sortierauswahl mit drei Optionen:

- **Werte:** Die Filme werden nach dem dargestellten Kennwert angeordnet. Bei der kürzesten Einstellungslänge stehen die kleinsten Werte zuerst, bei den anderen Diagrammen die größten.
- **Alphabetisch:** Die Filme werden nach ihren Titeln sortiert.
- **Chronologisch:** Die Filme werden nach ihrem Erscheinungsjahr sortiert.

Die Sortierung lässt sich für jedes Diagramm unabhängig verändern.

#### Darstellung

Die sechs Diagramme werden untereinander als gemeinsame Visualisierung direkt im Notebook angezeigt. Die Beschriftung, Farben und Abstände werden im Code festgelegt. Beim Bewegen des Mauszeigers über einen Balken wird dieser farblich hervorgehoben. Zusätzlich erscheinen das Erscheinungsjahr des Films und der jeweilige Kennwert als *Tooltip*.

Die Größe der Diagramme können Sie über `CHART_WIDTH` und `ROW_HEIGHT` im Abschnitt [Festlegen der Parameter](#festlegen-der-parameter) anpassen.

#### Interpretation

Beim Vergleich der Filme lohnt es sich, alle Kennwerte **im Zusammenhang** zu betrachten. Eine größere Anzahl erkannter Einstellungen bedeutet beispielsweise nicht zwangsläufig, dass ein Film schneller geschnitten ist, denn auch seine Gesamtlaufzeit spielt eine Rolle. Ein Unterschied zwischen durchschnittlicher und medianer Einstellungslänge kann darauf hindeuten, dass einzelne sehr lange Einstellungen den Durchschnitt beeinflussen.

**Wichtig:** Die Diagramme zeigen die Ergebnisse einer automatischen Einstellungserkennung. Übersehene oder fälschlich erkannte Einstellungswechsel können die Kennwerte beeinflussen. Die Visualisierungen bieten daher Anhaltspunkte für die Filmanalyse, ersetzen aber nicht die Überprüfung der zugrunde liegenden Annotationen.

In [6]:
# Use SVG for responsive sizing and German decimal and thousands separators.
# Hide the chart action menu in the notebook and the exported HTML file.
embed_options = {"actions": False, "renderer": "svg", "formatLocale": {
    "decimal": ",", "thousands": ".", "grouping": [3],
}}

# Assign one colour per metric.
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

# Initialise the list of charts.
charts = []

# Create one chart per metric directly from the film table.
for index, metric in enumerate(metrics):
    # Get the chart title, colour, and formatted hover column.
    chart_title = metrics[metric]
    color = colors[index]
    hover_column = f"{metric}_hover"

    # Give every chart its own sorting control.
    sort_name = f"sort_mode_{index}"
    sort_mode = alt.param(
        name=sort_name,
        # Set the default value.
        value="Werte",
        # Set the sort options.
        bind=alt.binding_radio(
            options=["Werte", "Alphabetisch", "Chronologisch"],
            name="Sortierung: ",
        ),
    )

    # Define the hovering condition: highlight a film under the pointer.
    hovered_film = alt.selection_point(
        fields=["video_name"],
        on="pointerover",
        clear="pointerout",
        empty=False,
    )

    # Sort minimum shot duration values in ascending order and other values in descending order.
    if metric == "min_duration_s":
        value_sort = f"datum.{metric}"
    else:
        value_sort = f"-datum.{metric}"

    # Create the base chart from the films DataFrame.
    base = alt.Chart(films).transform_calculate(
        # Switch between value, alphabetical, and chronological sorting.
        dynamic_sort=(
            f"{sort_name} === 'Alphabetisch' ? datum.alpha_rank "
            f": {sort_name} === 'Chronologisch' ? datum.year_rank : {value_sort}"
        )
    # Define the properties of the chart.
    ).encode(
        # Define the horizontal x-axis of the chart.
        x=alt.X(
            # Use the metric as quantitative ('Q') values.
            f"{metric}:Q",
            # Do not use an axis title.
            title=None,
            # Set the axis maximum to the largest metric value.
            scale=alt.Scale(domainMax=float(films[metric].max()), nice=False),
        ),

        # Define the vertical y-axis of the chart.
        y=alt.Y(
            # Use film titles as categories. N means nominal (categorical).
            "title:N",
            # Do not use an axis title.
            title=None,
            # Order the films using the calculated dynamic_sort field.
            sort=alt.EncodingSortField("dynamic_sort"),
        ),

        # Define the information shown when hovering over a bar.
        tooltip=[
            # Read the release year from the film table.
            # Use "Erscheinungsjahr" as the label in the tooltip.
            alt.Tooltip("release_year:N", title="Erscheinungsjahr"),

            # Read the formatted text from this metric's hover column.
            # Use "Wert" as the label in the tooltip.
            alt.Tooltip(f"{hover_column}:N", title="Wert"),
        ],
    ).properties(
        # Set the chart width using the CHART_WIDTH parameter.
        width=CHART_WIDTH,
        # Set the height of each film row using ROW_HEIGHT.
        height=alt.Step(ROW_HEIGHT),
    )

    # Set the color of the bars.
    bars = base.mark_bar(color=color).encode(
        # Increase the opacity when hovering over the bar.
        opacity=alt.condition(hovered_film, alt.value(1), alt.value(0.6)),
    )

    # Attach this chart's independent sorting and hover interactions.
    chart = bars.add_params(
        sort_mode, hovered_film,
    ).properties(
        # Add the left-aligned (anchor="start") title.
        title=alt.TitleParams(chart_title, anchor="start"),
    )
    charts.append(chart)

# Use a unique identifier to keep repeated notebook outputs separate.
chart_output_id = f"shots-{uuid4().hex}"

# Use CSS rules to style the visualisation for notebook and Jupyter Book display.
chart_styles = """<style>
/* Emphasise the overall heading. */
.shot-vis .shot-title {
    font-size: 17px;
    font-weight: bold;
    margin-bottom: 18px;
}

/* Leave space between charts. */
.shot-vis .shot-panel {
    margin-bottom: 28px;
}

/* Place the sorting controls below each chart.
   Override Altair's default layout with !important. */
.shot-vis .vega-embed {
    display: block !important;
    width: 100%;
    padding: 0;
}
</style>"""

# Start with the CSS rules for the visualisation.
chart_html = chart_styles

# Open the container holding all charts.
chart_html += f'<div id="{chart_output_id}" class="shot-vis">'

# Add the overall heading.
chart_html += '<div class="shot-title">Einstellungslängen und Einstellungsanzahl im Filmvergleich</div>'

# Convert each chart to HTML.
for index, chart in enumerate(charts):
    chart_content = chart.to_html(
        # Create an HTML fragment rather than a complete webpage.
        fullhtml=False,

        # Use Altair's loader for notebook and webpage display.
        template="universal",

        # Give each chart a unique HTML identifier.
        output_div=f"{chart_output_id}-{index}",

        # Apply the display settings defined earlier.
        embed_options=embed_options,
    )

    # Place the chart inside its own panel for spacing.
    chart_html += '<div class="shot-panel">'
    chart_html += chart_content
    chart_html += '</div>'

# Close the container holding all charts.
chart_html += '</div>'

# Display the assembled HTML in the notebook.
display(HTML(chart_html))

---
## Export der Ergebnisse

Die fertige Visualisierung wird nun als HTML-Datei gespeichert. Dadurch kann sie auch außerhalb des Jupyter Notebooks in einem Webbrowser betrachtet und weitergegeben werden. Die zuvor eingelesenen Metadaten-CSV-Dateien bleiben dabei unverändert.

### Export der Visualisierung als `.html`-Datei

Der folgende Codeblock speichert die sechs Balkendiagramme gemeinsam unter dem Namen `shot_metadata.html` im zuvor festgelegten Exportordner (`EXPORT_PATH`). Die interaktiven Sortierfelder und die zusätzlichen Informationen beim Überfahren der Balken bleiben in der exportierten Datei erhalten.

Dafür wird die bereits im Notebook angezeigte Visualisierung in eine vollständige HTML-Seite eingefügt und gespeichert. Beim erneuten Ausführen wird eine vorhandene Datei mit demselben Namen ersetzt.

**Hinweis:** Zum Laden der eingebundenen JavaScript-Bibliotheken wird beim Öffnen der HTML-Datei normalerweise eine Internetverbindung benötigt.

In [7]:
# Convert the export folder setting into a path.
export_directory = Path(EXPORT_PATH)

# Combine the folder path and filename.
output_html = export_directory / "shot_metadata.html"

# Build a complete HTML page containing the visualisation.
# The f-string inserts the chart_html created earlier.
html_document = f"""<!DOCTYPE html>
<html lang="de">
<head>
    <!-- Use UTF-8 to display German characters correctly. -->
    <meta charset="UTF-8">

    <!-- Adapt the page to the width of the browser window. -->
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <!-- Set the title shown in the browser tab. -->
    <title>Einstellungsmetadaten</title>
</head>

<!-- Remove the browser's default margin around the page content. -->
<body style="margin:0">
    {chart_html}
</body>
</html>
"""

# Save the page as an HTML file.
# UTF-8 ensures German characters are written correctly.
output_html.write_text(html_document, encoding="utf-8")

# Show where the file was saved.
print(f"Saved HTML visualisation to: {output_html}")

Saved HTML visualisation to: export_vis_shots/shot_metadata.html


### Herunterladen des Exportordners aus *Google Colab*

Dieser optionale Schritt ist nur für eine Ausführung in *Google Colab* vorgesehen. Der gesamte Exportordner in *Colab* wird zu einer `.zip`-Datei komprimiert und anschließend über die Browseroberfläche heruntergeladen. Der Download ist notwendig, da ansonsten die erstellten Visualisierungsdateien nach Beenden der Laufzeit bzw. von *Colab* immer gelöscht werden, ebenso wie alle in Colab hochgeladenen Dateien mit den Annotationsdaten.

In einer lokalen *Jupyter Notebook*-Umgebung bleiben die Dateien im anfangs im Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) definierten Export-Ordner erhalten und können von dort entnommen werden. Es ist daher kein zusätzlicher Download-Schritt erforderlich.


In [8]:
try:
    # The following import works only within a Google Colab runtime.
    from google.colab import files

    # Package the entire export directory as a ZIP archive.
    zip_path = f"{EXPORT_PATH}.zip"
    !zip -r "{zip_path}" "{EXPORT_PATH}"

    # Download the ZIP archive in the browser.
    files.download(zip_path)

except ImportError:
    # Skip this step when running locally.
    print("Download only works in Google Colab.")

Download only works in Google Colab.
